# 在 AMD Instinct™ GPU 上使用 verl 进行 Fully Async DAPO / GRPO 训练

本 tutorial 基于下面这份脚本逐项展开：

`verl/experimental/fully_async_policy/shell/dapo_7b_math_fsdp2_2_2.sh`

训练配置的核心是：

- **Model**: `Qwen/Qwen2.5-Math-7B`
- **Training engine**: FSDP2
- **Rollout engine**: vLLM async mode
- **RL advantage estimator**: GRPO
- **Reward manager**: DAPO
- **Dataset**: DAPO-Math-17k
- **Validation**: AIME 2024
- **Hardware layout**: 4 GPUs total
  - 2 GPUs for training
  - 2 GPUs for rollout
- **Fully async mode**:
  - `staleness_threshold=0.1`
  - `trigger_parameter_sync_step=4`
  - `require_batches=4`
  - `partial_rollout=True`

这个 notebook 的目标不是把 shell script 简单复制进一个 cell，而是把它拆成：

1. 为什么要使用 Fully Async RL
2. verl 的 Trainer / Rollouter 如何解耦
3. DAPO / GRPO 配置分别控制什么
4. 4 张 AMD GPU 如何进行 2 + 2 资源隔离
5. 如何准备数据集和 Qwen2.5-Math-7B
6. 如何检查训练前置条件
7. 如何启动、监控和停止训练
8. 如何理解 staleness、parameter sync 和 partial rollout

> **重要说明**
>
> 原脚本使用 `trainer.save_freq=-1`，因此默认**不会保存 checkpoint**。
> 本 tutorial 会忠实保留这个行为。最后会额外给出一个“可选扩展”，说明如果你想保存模型并做 inference，需要修改什么。

## 为什么要做 Fully Async RL Training？

传统 colocate RL 训练中，rollout 和 training 往往交替发生：

```text
Generate rollouts
      ↓
Wait
      ↓
Train actor
      ↓
Wait
      ↓
Sync weights
      ↓
Generate next rollouts
```

对于 reasoning model，response 长度差异可能非常大。

同一批 prompt 中：

```text
sample A:  500 tokens
sample B: 1200 tokens
sample C: 3800 tokens
sample D: 4096 tokens
```

短 response 已经生成完时，GPU 仍然可能需要等待最长的 response。

Fully Async 的核心思路是把 **Rollouter** 和 **Trainer** 放到不同 GPU 上，让二者并行：

```text
                    ┌────────────────────┐
Prompts ───────────▶│  Rollouter / vLLM  │  GPU 2-3
                    └─────────┬──────────┘
                              │ generated samples
                              ▼
                       ┌─────────────┐
                       │Message Queue│
                       └──────┬──────┘
                              │
                              ▼
                    ┌────────────────────┐
                    │ Trainer / FSDP2    │  GPU 0-1
                    └─────────┬──────────┘
                              │
                              │ updated parameters
                              ▼
                    Parameter Synchronizer
                              │
                              └──────────────▶ Rollouter
```

这样：

- Rollouter 可以持续生成新样本
- Trainer 可以持续消费已经生成好的样本
- 两边不需要每一步都互相等待
- parameter sync 只在设定的训练更新次数后发生
- 允许有限度地使用旧参数生成的样本，从而减少 pipeline bubble

### 这次训练使用的组件

| Layer | Setting |
|---|---|
| Base model | `Qwen/Qwen2.5-Math-7B` |
| Training dataset | `DAPO-Math-17k` |
| Validation dataset | `AIME 2024` |
| Advantage estimator | `grpo` |
| Reward manager | `dapo` |
| Actor strategy | `FSDP2` |
| Rollout backend | `vLLM` |
| Rollout mode | `async` |
| Responses per prompt | `16` |
| Total GPUs | `4` |
| Trainer GPUs | `2` |
| Rollout GPUs | `2` |
| Tensor parallel for rollout | `1` |
| Actor FSDP size | `2` |
| Max prompt length | `2048` |
| Max response length | `4096` |
| Partial rollout | `True` |
| Staleness threshold | `0.1` |

## Fully Async 中 verl 和 vLLM 分别负责什么？

在这个配置里，verl 不是把 vLLM 当成普通的 inference library 使用，而是把它作为独立的 rollout runtime。

```text
             Rollout side                             Training side

     ┌─────────────────────┐                 ┌─────────────────────────┐
     │ vLLM async servers  │                 │ FSDP2 Actor             │
     │                     │                 │                         │
     │ prompt              │                 │ consume trajectories    │
     │   ↓                 │                 │   ↓                     │
     │ generate n=16       │ ── samples ──▶  │ GRPO advantages         │
     │ responses           │                 │   ↓                     │
     └─────────────────────┘                 │ PPO-style clipped loss  │
                                             │   ↓                     │
                                             │ optimizer update        │
                                             └────────────┬────────────┘
                                                          │
                                                  parameter sync
                                                          │
                                                          ▼
                                                 vLLM gets new weights
```

这里最重要的区别是：

**rollout 和 training 不再使用同一组 GPU。**

脚本中：

```bash
NGPUS_PER_NODE=4
n_gpus_rollout=2
n_gpus_training=2
```

所以它是一个 **2 trainer + 2 rollout** 的资源隔离配置。

## Step 1: 检查 AMD GPU

首先确认当前环境能够看到 4 张 AMD GPU。

我们希望检查：

- ROCm / AMD GPU runtime 正常
- 至少有 4 张 GPU
- GPU memory 没有被其他训练任务大量占用

In [ ]:
!amd-smi

也可以从 PyTorch 角度再次确认 GPU 数量。

In [ ]:
import torch

hip_version = getattr(torch.version, "hip", None)
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0

print("PyTorch version :", torch.__version__)
print("HIP runtime     :", hip_version)
print("Visible GPUs    :", gpu_count)

for i in range(gpu_count):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

assert hip_version is not None, "Current PyTorch is not a ROCm/HIP build."
assert gpu_count >= 4, f"This tutorial expects at least 4 GPUs, but only {gpu_count} are visible."

print("\nOK: ROCm + 4 GPU environment is ready.")

## Step 2: 验证 verl 和 vLLM 环境

Fully Async Policy Trainer 依赖：

- `verl`
- `vllm`
- PyTorch ROCm build
- Ray

下面检查这些组件是否可以正常导入。

In [ ]:
import verl
import vllm
import ray
import torch

print("verl :", getattr(verl, "__version__", "unknown"), verl.__file__)
print("vLLM :", vllm.__version__)
print("Ray  :", ray.__version__)
print("HIP  :", torch.version.hip)

print("\nEnvironment import check passed.")

## Step 3: 配置本 tutorial 使用的路径

原始 shell script 默认使用：

```text
${HOME}/verl/
├── models/
│   └── Qwen2.5-Math-7B/
├── data/
│   ├── dapo-math-17k.parquet
│   └── aime-2024.parquet
└── ckpts/
    └── DAPO/
```

下面把这些路径统一定义出来。

如果你的 verl repo 不在 `/workspace/verl`，只需要修改 `VERL_DIR`。

In [ ]:
import os
from pathlib import Path

VERL_DIR = Path(os.environ.get("VERL_DIR", "/workspace/verl"))
RAY_DATA_HOME = Path(os.environ.get("RAY_DATA_HOME", str(Path.home() / "verl")))

MODEL_PATH = RAY_DATA_HOME / "models" / "Qwen2.5-Math-7B"
TRAIN_FILE = RAY_DATA_HOME / "data" / "dapo-math-17k.parquet"
TEST_FILE = RAY_DATA_HOME / "data" / "aime-2024.parquet"
os.environ["VERL_DIR"] = str(VERL_DIR)
os.environ["RAY_DATA_HOME"] = str(RAY_DATA_HOME)

CKPTS_DIR = (
    RAY_DATA_HOME
    / "ckpts"
    / "DAPO"
    / "DAPO-Qwen2.5-7b-MATH-0527a1-fsdp2-fully-async-2-2"
)

print("VERL_DIR      :", VERL_DIR)
print("RAY_DATA_HOME :", RAY_DATA_HOME)
print("MODEL_PATH    :", MODEL_PATH)
print("TRAIN_FILE    :", TRAIN_FILE)
print("TEST_FILE     :", TEST_FILE)
print("CKPTS_DIR     :", CKPTS_DIR)

## Step 4: 准备 DAPO-Math-17k 和 AIME 2024

脚本要求两个已经处理成 verl 格式的 parquet：

```text
~/verl/data/dapo-math-17k.parquet
~/verl/data/aime-2024.parquet
```

官方 DAPO recipe 使用的就是这两个数据文件：

- Training: `BytedTsinghua-SIA/DAPO-Math-17k`
- Validation: `BytedTsinghua-SIA/AIME-2024`

下面的 cell 会：

1. 创建 `~/verl/data`
2. 如果文件已经存在则跳过
3. 如果不存在则直接下载官方准备好的 parquet

In [ ]:
%%bash
set -euo pipefail

RAY_DATA_HOME="${RAY_DATA_HOME:-${HOME}/verl}"
DATA_DIR="${RAY_DATA_HOME}/data"

TRAIN_FILE="${DATA_DIR}/dapo-math-17k.parquet"
TEST_FILE="${DATA_DIR}/aime-2024.parquet"

mkdir -p "${DATA_DIR}"

if [ -f "${TRAIN_FILE}" ]; then
  echo "Training data already exists: ${TRAIN_FILE}"
else
  echo "Downloading DAPO-Math-17k..."
  wget -O "${TRAIN_FILE}" \
    "https://huggingface.co/datasets/BytedTsinghua-SIA/DAPO-Math-17k/resolve/main/data/dapo-math-17k.parquet?download=true"
fi

if [ -f "${TEST_FILE}" ]; then
  echo "Validation data already exists: ${TEST_FILE}"
else
  echo "Downloading AIME-2024..."
  wget -O "${TEST_FILE}" \
    "https://huggingface.co/datasets/BytedTsinghua-SIA/AIME-2024/resolve/main/data/aime-2024.parquet?download=true"
fi

echo
ls -lh "${TRAIN_FILE}" "${TEST_FILE}"

### 看一眼数据长什么样

DAPO 数据已经包含 verl RL trainer 所需要的结构，例如：

- `prompt`
- `reward_model`
- `data_source`
- `ability`
- `extra_info`

训练时脚本使用：

```text
data.prompt_key=prompt
```

所以模型实际读取的是 `prompt` 字段。

In [ ]:
import pandas as pd
from pathlib import Path

data_home = Path.home() / "verl" / "data"
train_file = data_home / "dapo-math-17k.parquet"
test_file = data_home / "aime-2024.parquet"

train_df = pd.read_parquet(train_file)
test_df = pd.read_parquet(test_file)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("\nTrain columns:", train_df.columns.tolist())

sample = train_df.iloc[0]

print("\n--- prompt ---")
print(sample["prompt"])

print("\n--- reward_model ---")
print(sample["reward_model"])

## Step 5: 准备 Qwen2.5-Math-7B

原脚本使用：

```text
~/verl/models/Qwen2.5-Math-7B
```

并且特别要求把模型的：

```json
"max_position_embeddings": 32768
```

设置为 `32768`。

虽然这个 tutorial 的：

```text
max_prompt_length   = 2048
max_response_length = 4096
```

加起来只有 `6144` tokens，但我们仍然保持和训练 recipe 完全一致的模型配置。

In [ ]:
%%bash
set -euo pipefail

RAY_DATA_HOME="${RAY_DATA_HOME:-${HOME}/verl}"
MODEL_PATH="${RAY_DATA_HOME}/models/Qwen2.5-Math-7B"

mkdir -p "$(dirname "${MODEL_PATH}")"

if [ -f "${MODEL_PATH}/config.json" ]; then
  echo "Model already exists at ${MODEL_PATH}, skipping download."
else
  echo "Downloading Qwen/Qwen2.5-Math-7B..."
  hf download Qwen/Qwen2.5-Math-7B --local-dir "${MODEL_PATH}"
fi

python3 - <<'PY'
import json
from pathlib import Path
import os

model_path = Path(os.environ.get(
    "MODEL_PATH",
    str(Path.home() / "verl" / "models" / "Qwen2.5-Math-7B")
))
config_path = model_path / "config.json"

with config_path.open() as f:
    config = json.load(f)

before = config.get("max_position_embeddings")
config["max_position_embeddings"] = 32768

with config_path.open("w") as f:
    json.dump(config, f, indent=2)
    f.write("\n")

print("config.json:", config_path)
print("max_position_embeddings:", before, "->", config["max_position_embeddings"])
PY

## Step 6: 先理解这份脚本到底在训练什么

### 6.1 GRPO advantage estimator

脚本使用：

```text
algorithm.adv_estimator=grpo
actor_rollout_ref.rollout.n=16
```

也就是说，对于一个 prompt，Rollouter 会生成 **16 条 response**：

```text
prompt
 ├── response 1
 ├── response 2
 ├── response 3
 ├── ...
 └── response 16
```

这些 response 会得到各自的 reward。

GRPO 的关键思想是：

> 不单独训练一个 critic 来估计 value，而是在同一个 prompt 的 group 内比较 reward，计算相对 advantage。

因此：

```text
n_resp_per_prompt = 16
```

就是 GRPO 中非常关键的 **group size**。

### 6.2 这是不是“完整 DAPO”？

这份脚本：

```text
algorithm.adv_estimator=grpo
reward.reward_manager.name=dapo
clip_ratio_low=0.2
clip_ratio_high=0.28
clip_ratio_c=10.0
```

因此它使用了明显的 DAPO-style 配置：

- asymmetric / decoupled clipping
- DAPO reward manager
- overlong reward shaping
- token-level loss aggregation

但是脚本中**没有显式开启 DAPO 的 dynamic sampling / filter groups 配置**。

所以更准确地说，这个 tutorial 展示的是：

> **基于 DAPO recipe 的 GRPO + Fully Async training setup**

而不是去声称脚本包含 DAPO 论文中每一个组件。

### 6.3 KL 为什么全部关闭？

脚本里：

```text
algorithm.use_kl_in_reward=False
algorithm.kl_ctrl.kl_coef=0.0

actor_rollout_ref.actor.use_kl_loss=False
actor_rollout_ref.actor.kl_loss_coef=0.0
```

也就是说：

```text
Reward 中没有 KL penalty
+
Actor loss 中也没有 KL loss
```

这和很多 PPO / GRPO recipe 不同。

这里主要依赖：

- clipped policy objective
- reward signal
- GRPO group-relative advantage

来约束更新。

### 6.4 DAPO 的 clipping 配置

脚本中：

| Config | Value |
|---|---:|
| `clip_ratio_low` | `0.20` |
| `clip_ratio_high` | `0.28` |
| `clip_ratio_c` | `10.0` |

这意味着正负方向的 clipping 不是完全对称的。

普通 PPO 经常写成类似：

```text
clip_ratio = 0.2
```

而这里拆成：

```text
low  = 0.20
high = 0.28
```

允许 policy 在一侧具有更大的更新空间。

### 6.5 Response length 与 Overlong Buffer

脚本：

```text
max_prompt_length   = 2048
max_response_length = 4096

overlong_buffer_len      = 4096
overlong_penalty_factor  = 1.0
```

reward manager 配置：

```text
reward.reward_manager.name=dapo
reward.reward_kwargs.overlong_buffer_cfg.enable=True
reward.reward_kwargs.overlong_buffer_cfg.len=4096
reward.reward_kwargs.overlong_buffer_cfg.penalty_factor=1.0
```

它的目的之一是处理 reasoning model 容易不断拉长 response 的问题。

如果模型为了探索而持续生成超长回答，reward 可以对这种行为施加额外约束。

### 6.6 Actor / Reference / Rollout 的显存策略

| Component | Config | Meaning |
|---|---|---|
| Actor | `fsdp2` | 训练模型使用 FSDP2 |
| Actor | `fsdp_size=2` | Actor 在 2 张 trainer GPU 上分片 |
| Actor param offload | `False` | Actor 参数留在 GPU |
| Actor optimizer offload | `False` | Optimizer state 留在 GPU |
| Reference model | `param_offload=True` | Reference 参数允许 offload 到 CPU |
| Rollout | `vllm` | 使用 vLLM 生成 response |
| Rollout TP | `1` | 每个 rollout engine 不做 tensor parallel |
| vLLM GPU util | `0.80` | vLLM 允许使用较高比例的 GPU memory |

### 6.7 Dynamic batch 不是“动态改变 prompt batch size”

脚本启用：

```text
actor_rollout_ref.actor.use_dynamic_bsz=True
actor_rollout_ref.ref.log_prob_use_dynamic_bsz=True
actor_rollout_ref.rollout.log_prob_use_dynamic_bsz=True
```

这里的 dynamic batch 主要按 **token budget** 组织 micro-batch。

因为 reasoning response 的长度差异很大：

```text
sample 1:  800 tokens
sample 2: 4000 tokens
```

固定按“样本数量”分 batch，GPU workload 会非常不均衡。

所以脚本用：

```text
actor_ppo_max_token_len_per_gpu
infer_ppo_max_token_len_per_gpu
```

来控制每张 GPU 一次最多处理多少 token。

In [ ]:
max_prompt_length = 2048
max_response_length = 4096

actor_ppo_max_token_len = (max_prompt_length + max_response_length) * 2
infer_ppo_max_token_len = (max_prompt_length + max_response_length) * 3
vllm_max_num_batched_tokens = max_prompt_length + max_response_length

print("prompt + response max length :", max_prompt_length + max_response_length)
print("actor token budget / GPU     :", actor_ppo_max_token_len)
print("ref/rollout token budget/GPU :", infer_ppo_max_token_len)
print("vLLM max batched tokens      :", vllm_max_num_batched_tokens)

## Step 7: 理解 Fully Async 的四个关键参数

这是整个 tutorial 最重要的一部分。

脚本：

```text
ppo_mini_batch_size         = 32
require_batches             = 4
trigger_parameter_sync_step = 4
staleness_threshold         = 0.1
partial_rollout             = True
```

### 7.1 `require_batches=4`

Trainer 不会一收到一条 rollout 就更新模型。

它会等待：

```text
require_batches × ppo_mini_batch_size
```

所以：

```text
4 × 32 = 128
```

Trainer 每次取 **128 个 streaming samples** 后进行一次 local training update。

由于每个 prompt 有：

```text
rollout.n = 16
```

所以一个 prompt 会对应一个 16-response group。

### 7.2 `trigger_parameter_sync_step=4`

Trainer 连续做 4 次 local update 后，才向 Rollouter 同步一次新参数。

因此两次 parameter sync 之间，Trainer 处理的 sample 数量是：

```text
trigger_parameter_sync_step
× require_batches
× ppo_mini_batch_size
```

也就是：

```text
4 × 4 × 32 = 512
```

这也是为什么脚本的：

```text
total_rollout_steps = 512 × 100 = 51200
```

可以很自然地理解为大约对应 **100 个 512-sample 的逻辑训练区间**。

In [ ]:
ppo_mini_batch_size = 32
require_batches = 4
trigger_parameter_sync_step = 4
rollout_n = 16
total_rollout_steps = 512 * 100

samples_per_local_update = require_batches * ppo_mini_batch_size
samples_between_sync = (
    trigger_parameter_sync_step
    * require_batches
    * ppo_mini_batch_size
)

print("Samples / local update      :", samples_per_local_update)
print("Local updates / param sync  :", trigger_parameter_sync_step)
print("Samples between param sync  :", samples_between_sync)
print("Responses per prompt        :", rollout_n)
print("Total rollout samples       :", total_rollout_steps)
print("Equivalent 512-sample units :", total_rollout_steps // samples_between_sync)

### 7.3 `staleness_threshold=0.1`

Fully Async 的代价是：

> Trainer 当前正在训练的参数版本，可能已经比生成某些 rollout 时使用的参数更新。

这些 rollout 就是 **stale samples**。

```text
Rollouter uses policy v10
       ↓
generates sample
       ↓

Meanwhile:

Trainer updates:
v10 → v11 → v12

       ↓
sample generated by v10 reaches Trainer
```

`staleness_threshold=0.1` 允许系统保留有限比例的这种旧样本。

它让 pipeline 更连续，但同时不会无限制地允许 off-policy sample 堆积。

### 7.4 `partial_rollout=True`

假设 parameter sync 即将发生，但是 vLLM 正在生成一条很长的 response：

```text
generated:
token 1
token 2
...
token 1800
...
target max = 4096
```

如果没有 partial rollout，系统可能需要：

```text
等这条 response 完整结束
→ 再 parameter sync
```

启用 partial rollout 后，可以：

```text
暂停正在生成的 rollout
→ parameter sync
→ 用新参数恢复后续 generation
```

这样可以减少等待长尾 response 的时间。

在这个脚本中：

```text
staleness_threshold=0.1
partial_rollout=True
```

两者是配套使用的。

## Step 8: Training 前做一次 Preflight Check

正式启动前检查：

- verl repo 存在
- 训练数据存在
- AIME validation 数据存在
- 模型存在
- `max_position_embeddings=32768`
- GPU 数量至少为 4

In [ ]:
import json
from pathlib import Path
import torch
import os

verl_dir = Path(os.environ.get("VERL_DIR", "/workspace/verl"))
ray_data_home = Path(os.environ.get("RAY_DATA_HOME", str(Path.home() / "verl")))

model_path = ray_data_home / "models" / "Qwen2.5-Math-7B"
train_file = ray_data_home / "data" / "dapo-math-17k.parquet"
test_file = ray_data_home / "data" / "aime-2024.parquet"

checks = {
    "verl repo": verl_dir.exists(),
    "model": model_path.exists(),
    "train parquet": train_file.exists(),
    "validation parquet": test_file.exists(),
    "4 GPUs": torch.cuda.device_count() >= 4,
}

for name, ok in checks.items():
    print(f"{name:<22}: {'OK' if ok else 'MISSING'}")

assert all(checks.values()), "Preflight check failed."

with (model_path / "config.json").open() as f:
    config = json.load(f)

assert config.get("max_position_embeddings") == 32768, (
    "Please set max_position_embeddings=32768 in Qwen2.5-Math-7B/config.json"
)

print("\nAll preflight checks passed.")

## Step 9: 启动 2 + 2 Fully Async DAPO / GRPO Training

下面这个 cell 基本保持原 shell script 的训练参数不变，只做两件适合 notebook 的改造：

1. 使用 `nohup` 放到后台运行，避免 notebook cell 一直被占用
2. 把 stdout / stderr 写入独立的 `train_log.txt`

训练本身仍然是：

```text
2 trainer GPUs + 2 rollout GPUs
FSDP2 + vLLM
GRPO + DAPO reward
Fully Async
```

> 原脚本的 `trainer.save_freq=-1` 也会保留，因此这次默认不会保存 checkpoint。

In [ ]:
%%bash
set -euo pipefail

VERL_DIR="${VERL_DIR:-/workspace/verl}"
RAY_DATA_HOME="${RAY_DATA_HOME:-${HOME}/verl}"

project_name="DAPO"
exp_name="DAPO-Qwen2.5-7b-MATH-0527a1-fsdp2-fully-async-2-2"

MODEL_PATH="${MODEL_PATH:-${RAY_DATA_HOME}/models/Qwen2.5-Math-7B}"
CKPTS_DIR="${CKPTS_DIR:-${RAY_DATA_HOME}/ckpts/${project_name}/${exp_name}}"
TRAIN_FILE="${TRAIN_FILE:-${RAY_DATA_HOME}/data/dapo-math-17k.parquet}"
TEST_FILE="${TEST_FILE:-${RAY_DATA_HOME}/data/aime-2024.parquet}"

rollout_mode="async"
rollout_name="vllm"

export VLLM_USE_V1=1
return_raw_chat="True"

# Algorithm
adv_estimator="grpo"

use_kl_in_reward=False
kl_coef=0.0
use_kl_loss=False
kl_loss_coef=0.0

clip_ratio_low=0.2
clip_ratio_high=0.28

# Length
max_prompt_length=$((1024 * 2))
max_response_length=$((1024 * 4))

enable_overlong_buffer=True
overlong_buffer_len=$((1024 * 4))
overlong_penalty_factor=1.0

loss_agg_mode="token-mean"

# Sampling
temperature=1.0
top_p=1.0
top_k=-1
val_top_p=0.7

# Performance
use_dynamic_bsz=True
actor_ppo_max_token_len=$(((max_prompt_length + max_response_length) * 2))
infer_ppo_max_token_len=$(((max_prompt_length + max_response_length) * 3))

ref_offload=True
actor_offload=False

gen_tp=1
sp_size=1
fsdp_size=2

# Fully async resource layout
NNODES="${NNODES:-1}"
NGPUS_PER_NODE="${NGPUS_PER_NODE:-4}"

n_gpus_rollout="${N_GPUS_ROLLOUT:-2}"
n_gpus_training=$((NGPUS_PER_NODE - n_gpus_rollout))

train_prompt_bsz=0
gen_prompt_bsz=1
n_resp_per_prompt=16
train_prompt_mini_bsz=32

total_rollout_steps=$((512 * 100))
test_freq=10

staleness_threshold=0.1
trigger_parameter_sync_step=4
require_batches=4
partial_rollout=True

TIMESTAMP=$(date +%Y%m%d.%H%M%S)
RUN_DIR="${RAY_DATA_HOME}/tutorial_runs/${exp_name}_${TIMESTAMP}"
mkdir -p "${RUN_DIR}"

echo "${RUN_DIR}" > /tmp/verl_fully_async_last_run

cd "${VERL_DIR}"

nohup python -m verl.experimental.fully_async_policy.fully_async_main \
    data.train_files="${TRAIN_FILE}" \
    data.val_files="${TEST_FILE}" \
    data.prompt_key=prompt \
    data.truncation='left' \
    data.max_prompt_length=${max_prompt_length} \
    data.max_response_length=${max_response_length} \
    data.train_batch_size=${train_prompt_bsz} \
    data.gen_batch_size=${gen_prompt_bsz} \
    data.return_raw_chat=${return_raw_chat} \
    actor_rollout_ref.rollout.n=${n_resp_per_prompt} \
    algorithm.adv_estimator=${adv_estimator} \
    algorithm.use_kl_in_reward=${use_kl_in_reward} \
    algorithm.kl_ctrl.kl_coef=${kl_coef} \
    actor_rollout_ref.actor.fsdp_config.strategy=fsdp2 \
    critic.strategy=fsdp2 \
    actor_rollout_ref.actor.use_kl_loss=${use_kl_loss} \
    actor_rollout_ref.actor.kl_loss_coef=${kl_loss_coef} \
    actor_rollout_ref.actor.clip_ratio_low=${clip_ratio_low} \
    actor_rollout_ref.actor.clip_ratio_high=${clip_ratio_high} \
    actor_rollout_ref.actor.clip_ratio_c=10.0 \
    actor_rollout_ref.model.use_remove_padding=True \
    actor_rollout_ref.hybrid_engine=False \
    +actor_rollout_ref.model.override_config.max_position_embeddings=32768 \
    actor_rollout_ref.actor.use_dynamic_bsz=${use_dynamic_bsz} \
    actor_rollout_ref.ref.log_prob_use_dynamic_bsz=${use_dynamic_bsz} \
    actor_rollout_ref.rollout.log_prob_use_dynamic_bsz=${use_dynamic_bsz} \
    actor_rollout_ref.actor.ppo_max_token_len_per_gpu=${actor_ppo_max_token_len} \
    actor_rollout_ref.ref.log_prob_max_token_len_per_gpu=${infer_ppo_max_token_len} \
    actor_rollout_ref.rollout.log_prob_max_token_len_per_gpu=${infer_ppo_max_token_len} \
    actor_rollout_ref.model.path="${MODEL_PATH}" \
    actor_rollout_ref.actor.optim.lr=1e-6 \
    actor_rollout_ref.actor.optim.lr_warmup_steps=10 \
    actor_rollout_ref.actor.optim.weight_decay=0.1 \
    actor_rollout_ref.actor.ppo_mini_batch_size=${train_prompt_mini_bsz} \
    actor_rollout_ref.actor.fsdp_config.param_offload=${actor_offload} \
    actor_rollout_ref.actor.fsdp_config.optimizer_offload=${actor_offload} \
    actor_rollout_ref.actor.entropy_coeff=0 \
    actor_rollout_ref.actor.grad_clip=1.0 \
    actor_rollout_ref.actor.loss_agg_mode=${loss_agg_mode} \
    actor_rollout_ref.actor.ulysses_sequence_parallel_size=${sp_size} \
    actor_rollout_ref.rollout.gpu_memory_utilization=0.80 \
    actor_rollout_ref.rollout.tensor_model_parallel_size=${gen_tp} \
    actor_rollout_ref.rollout.enable_chunked_prefill=True \
    actor_rollout_ref.rollout.max_num_batched_tokens=$((max_prompt_length + max_response_length)) \
    actor_rollout_ref.rollout.temperature=${temperature} \
    actor_rollout_ref.rollout.top_p=${top_p} \
    actor_rollout_ref.rollout.top_k=${top_k} \
    actor_rollout_ref.rollout.val_kwargs.temperature=${temperature} \
    actor_rollout_ref.rollout.val_kwargs.top_p=${val_top_p} \
    actor_rollout_ref.rollout.val_kwargs.top_k=${top_k} \
    actor_rollout_ref.rollout.val_kwargs.do_sample=True \
    actor_rollout_ref.rollout.val_kwargs.n=1 \
    actor_rollout_ref.rollout.calculate_log_probs=True \
    actor_rollout_ref.ref.fsdp_config.param_offload=${ref_offload} \
    actor_rollout_ref.ref.ulysses_sequence_parallel_size=${sp_size} \
    actor_rollout_ref.actor.fsdp_config.fsdp_size=${fsdp_size} \
    actor_rollout_ref.rollout.name=${rollout_name} \
    actor_rollout_ref.rollout.mode=${rollout_mode} \
    reward.reward_manager.name=dapo \
    +reward.reward_kwargs.overlong_buffer_cfg.enable=${enable_overlong_buffer} \
    +reward.reward_kwargs.overlong_buffer_cfg.len=${overlong_buffer_len} \
    +reward.reward_kwargs.overlong_buffer_cfg.penalty_factor=${overlong_penalty_factor} \
    +reward.reward_kwargs.overlong_buffer_cfg.log=False \
    +reward.reward_kwargs.max_resp_len=${max_response_length} \
    trainer.logger="['console','tensorboard']" \
    trainer.project_name="${project_name}" \
    trainer.experiment_name="${exp_name}" \
    trainer.val_before_train=True \
    trainer.save_freq=-1 \
    trainer.default_local_dir="${CKPTS_DIR}" \
    trainer.resume_mode=auto \
    trainer.nnodes="${NNODES}" \
    trainer.n_gpus_per_node="${n_gpus_training}" \
    rollout.nnodes="${NNODES}" \
    rollout.n_gpus_per_node="${n_gpus_rollout}" \
    rollout.total_rollout_steps="${total_rollout_steps}" \
    trainer.total_epochs=10 \
    trainer.test_freq="${test_freq}" \
    async_training.staleness_threshold="${staleness_threshold}" \
    async_training.trigger_parameter_sync_step="${trigger_parameter_sync_step}" \
    async_training.require_batches="${require_batches}" \
    async_training.partial_rollout="${partial_rollout}" \
    > "${RUN_DIR}/train_log.txt" 2>&1 &

PID=$!
echo "${PID}" > "${RUN_DIR}/train.pid"
disown

echo "Fully async training started."
echo "PID      : ${PID}"
echo "Run dir  : ${RUN_DIR}"
echo "Log file : ${RUN_DIR}/train_log.txt"
echo
echo "Resource split:"
echo "  trainer GPUs : ${n_gpus_training}"
echo "  rollout GPUs : ${n_gpus_rollout}"

### 启动后发生了什么？

训练启动后，并不是：

```text
rollout → train → rollout → train
```

而是两个长期运行的 pipeline：

```text
GPU 0-1
Trainer:
consume → update → consume → update → ...

GPU 2-3
Rollouter:
generate → generate → generate → generate → ...
```

中间通过：

```text
MessageQueue
+
ParameterSynchronizer
```

连接。

每完成：

```text
4 local updates
```

Trainer 会触发一次 parameter synchronization。

## Step 10: 实时查看训练日志

下面的 cell 每 3 秒刷新一次最近的日志。

停止这个 cell **不会停止训练**。

在 Jupyter / JupyterLab 中点击 Stop / Interrupt 即可停止查看。

In [ ]:
import time
import subprocess
from pathlib import Path
from IPython.display import clear_output

run_dir = Path(Path("/tmp/verl_fully_async_last_run").read_text().strip())
log_file = run_dir / "train_log.txt"
pid_file = run_dir / "train.pid"
pid = int(pid_file.read_text().strip())

def alive(process_id: int) -> bool:
    return Path(f"/proc/{process_id}").exists()

try:
    while True:
        result = subprocess.run(
            ["tail", "-n", "80", str(log_file)],
            capture_output=True,
            text=True,
        )

        clear_output(wait=True)
        status = "RUNNING" if alive(pid) else "EXITED"

        print(f"[{status}] PID={pid}")
        print(f"log={log_file}")
        print("-" * 100)
        print(result.stdout)

        if not alive(pid):
            break

        time.sleep(3)

except KeyboardInterrupt:
    print("\nStopped watching logs. Training continues in background.")

## 训练时重点看哪些 Fully Async 指标？

Fully Async 和普通 colocate training 最大的区别，是除了 reward / loss 之外，还要观察 **pipeline efficiency**。

比较重要的指标包括：

| Metric | 关注点 |
|---|---|
| `trainer/idle_ratio` | Trainer 有多少时间在等 rollout |
| `rollouter/idle_ratio` | Rollouter 有多少时间在等 Trainer / sync |
| `fully_async/count/stale_samples_processed` | 使用了多少 stale sample |
| `fully_async/count/stale_trajectory_processed` | stale trajectory 数量 |
| `fully_async/partial/total_partial_num` | partial rollout 数量 |
| `fully_async/partial/partial_ratio` | partial rollout 占比 |
| `fully_async/partial/max_partial_span` | partial sample 跨过了多少个参数版本 |

理想状态并不是简单追求某一个指标等于 0。

真正需要看的是：

```text
Trainer idle 是否很高？
Rollouter idle 是否很高？
stale sample 是否不断增长？
partial rollout 是否有效减少同步等待？
reward / validation 是否保持稳定？
```

### 如何根据 idle ratio 调整 2 + 2 资源分配？

当前脚本：

```text
Trainer  : 2 GPUs
Rollouter: 2 GPUs
```

如果观察到：

```text
trainer/idle_ratio 很高
rollouter/idle_ratio 很低
```

说明：

> Trainer 经常吃不到数据，rollout 相对太慢。

通常应该考虑给 rollout 更多资源。

反过来：

```text
rollouter/idle_ratio 很高
trainer/idle_ratio 很低
```

说明：

> rollout 生产能力有富余，而 Trainer 消费 / 更新较慢。

则可以考虑增加 training resource。

这正是 Fully Async resource isolation 的价值：Trainer 和 Rollouter 的 GPU 数量可以独立调节。

## TensorBoard

脚本设置：

```text
trainer.logger=['console','tensorboard']
```

所以你也可以通过 TensorBoard 查看训练曲线。

如果 event 文件写在 checkpoint / experiment 目录附近，可以启动：

```bash
tensorboard --logdir ~/verl --port 6006 --bind_all
```

然后在浏览器打开对应的 6006 端口。

In [ ]:
%%bash
echo "Search TensorBoard event files:"
find "${HOME}/verl" -type f -name "events.out.tfevents.*" 2>/dev/null | tail -20

## 如何停止当前训练任务

Fully Async 会启动 Ray worker、vLLM server 等多个进程。

因此只 kill notebook 启动的 Python PID，有时不足以立即释放 GPU。

下面的 cell 会：

1. 给 launcher 发送 `TERM`
2. 必要时发送 `KILL`
3. 清理本次实验残留的 fully async / Ray / vLLM 进程

> 如果这台机器上同时运行了其他 verl / Ray / vLLM 工作负载，不要直接使用这个清理 cell。

In [ ]:
%%bash
set -euo pipefail

RUN_DIR=$(cat /tmp/verl_fully_async_last_run)
PID=$(cat "${RUN_DIR}/train.pid")

echo "Stopping PID=${PID}"
kill -TERM "${PID}" 2>/dev/null || true

for _ in 1 2 3 4 5; do
  [ ! -d "/proc/${PID}" ] && break
  sleep 1
done

if [ -d "/proc/${PID}" ]; then
  kill -KILL "${PID}" 2>/dev/null || true
fi

echo
echo "Launcher stop request completed."
echo
echo "If GPU memory is still occupied, inspect processes first:"
echo "  pgrep -af 'fully_async_main|ray::|vllm'"
echo
echo "Do NOT blindly pkill shared Ray/vLLM processes on a multi-user machine."

## Optional: 如果想保存 checkpoint 并做 inference

原脚本：

```text
trainer.save_freq=-1
```

代表：

> **不保存 checkpoint。**

因此如果你的目标只是做：

- Fully Async 性能验证
- training convergence / AIME validation
- throughput 对比
- idle ratio / stale sample 分析

保持 `-1` 没问题。

但如果还想在训练后加载 actor 做 inference，需要把训练命令中的：

```text
trainer.save_freq=-1
```

改成例如：

```text
trainer.save_freq=10
```

然后重新启动训练。

### FSDP2 checkpoint 合并思路

有 checkpoint 后，通常可以把 actor 的 FSDP shard 合并成 Hugging Face 格式：

```bash
python -m verl.model_merger merge \
  --backend fsdp \
  --local_dir <checkpoint>/actor \
  --target_dir <checkpoint>/actor/merged_hf
```

之后再使用：

```python
AutoTokenizer.from_pretrained(...)
AutoModelForCausalLM.from_pretrained(...)
```

加载合并后的模型。

这个步骤没有放进默认执行路径，因为它会改变原始脚本 `save_freq=-1` 的行为。

## 最后把整个训练流程串起来

```text
DAPO-Math-17k prompts
        │
        ▼
┌─────────────────────────┐
│ Rollouter: vLLM async   │
│ GPUs: 2                 │
│ n = 16                  │
│ temp = 1.0              │
│ top_p = 1.0             │
└───────────┬─────────────┘
            │
            │ streaming trajectories
            ▼
      ┌──────────────┐
      │ MessageQueue │
      └───────┬──────┘
              │
              │ 4 × 32 = 128 samples
              ▼
┌─────────────────────────┐
│ Trainer: FSDP2          │
│ GPUs: 2                 │
│ GRPO advantage          │
│ DAPO reward             │
│ clip: 0.20 / 0.28       │
└───────────┬─────────────┘
            │
            │ 4 local updates
            ▼
┌─────────────────────────┐
│ Parameter Synchronizer  │
└───────────┬─────────────┘
            │
            └──────────────▶ Rollouter gets newer weights

staleness_threshold = 0.1
partial_rollout = True
```

一句话总结：

> **这份配置用 2 张 GPU 持续训练、2 张 GPU 持续生成，通过 streaming queue、有限 staleness 和 partial rollout 把 rollout 与 actor update 尽可能重叠，从而减少 reasoning RL 中因为长尾 response 和频繁同步造成的 GPU idle time。**

## 建议你在这个 tutorial 之后做的三个实验

### Experiment 1 — Fully Async vs Colocate

保持：

- model
- dataset
- rollout.n
- sequence length
- total rollout samples

尽量一致，对比：

```text
wall-clock time
trainer idle ratio
rollouter idle ratio
AIME validation
```

### Experiment 2 — 调整 Staleness

分别尝试：

```text
0
0.1
0.5
```

观察：

```text
throughput
stale samples
validation accuracy
response length
```

### Experiment 3 — 调整 Trainer / Rollouter GPU Ratio

在更多 GPU 的机器上尝试：

```text
2 trainer + 2 rollout
4 trainer + 4 rollout
2 trainer + 6 rollout
6 trainer + 2 rollout
```

用：

```text
trainer/idle_ratio
rollouter/idle_ratio
```

判断系统瓶颈到底在 generation 还是 training。

## References

- Training script used by this tutorial  
  `https://github.com/Vivicai1005/verl/blob/rocm/verl/experimental/fully_async_policy/shell/dapo_7b_math_fsdp2_2_2.sh`

- Reference workshop notebook  
  `https://github.com/Vivicai1005/rl-workshop/blob/main/verl_qwen3_4b_lora_grpo_workshop.ipynb`

- verl Fully Async Policy Trainer  
  `https://github.com/verl-project/verl/blob/main/docs/advance/fully_async.md`

- DAPO dataset  
  `https://huggingface.co/datasets/BytedTsinghua-SIA/DAPO-Math-17k`

- AIME 2024 validation dataset  
  `https://huggingface.co/datasets/BytedTsinghua-SIA/AIME-2024`